In [15]:
# imports
import os, math, json, random
import torch
import torch.nn as nn
from torch.utils.data import random_split
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv, global_mean_pool
import torch.nn.functional as F
from itertools import product
from sklearn.model_selection import KFold
import numpy as np
import xgboost as xgb

In [2]:
# dataset
NODE_FEATURE_DIM = 10
EDGE_FEATURE_DIM = 4

OP_TYPES = ['kn_input_op', 
            'kn_output_op', 
            'kn_add_op', 
            'kn_div_op',
            'kn_exp_op', 
            'kn_matmul_op', 
            'kn_mul_op',
            'kn_pow_op', 
            'kn_reduction_2_op', 
            'kn_sqrt_op']

def one_hot_op_type(op_type):
    out = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    out[OP_TYPES.index(op_type)] = 1.0
    return out

def compute_flops(op_type, input_tensors):
    if op_type == 'kn_matmul_op':
        M = input_tensors[0][-2]
        K = input_tensors[0][-1]
        N = input_tensors[1][-1]
        return 2 * M * N * K
    elif op_type in ['kn_add_op', 
                     'kn_div_op', 
                     'kn_exp_op', 
                     'kn_mul_op', 
                     'kn_pow_op', 
                     'kn_sqrt_op', 
                     'kn_reduction_2_op']:
        num_elements = 1
        for dim in input_tensors[0]:
            num_elements *= dim
        return num_elements
    elif op_type in ['kn_input_op', 'kn_output_op']:
        return 0
    else:
        raise ValueError(f"Unknown op type: {op_type}")

def get_node_features(node):
    # one-hot op type
    h_op_type = one_hot_op_type(node['op_type'])
    
    # in tensor dimensions
    in_tensors = []
    h_in_dims = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(node['input_tensors']):
        in_tensors.append(t['dim'])
        for j, dim in enumerate(t['dim']):
            h_in_dims[j + (i * 4)] = dim
    
    # out tensor dimensions
    out_tensors = []
    h_out_dims = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(node['output_tensors']):
        out_tensors.append(t['dim'])
        for j, dim in enumerate(t['dim']):
            h_out_dims[j + (i * 4)] = dim
    
    # in tensor sizes
    h_in_size = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(in_tensors):
        h_in_size[i] = math.prod(t)
    
    # out tensor sizes
    h_out_size = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(out_tensors):
        h_out_size[i] = math.prod(t)

    h_out_size = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    for i, t in enumerate(out_tensors):
        h_out_size[i] = math.prod(t)

    # computation cost in FLOPs
    flops = compute_flops(node['op_type'], in_tensors)
    h_flops = torch.zeros(NODE_FEATURE_DIM, dtype=torch.float)
    h_flops[0] = flops

    return torch.vstack([h_op_type, h_in_dims, h_out_dims, h_in_size, h_out_size, h_flops])

def get_edge_features(tensor):
    h_edge = torch.zeros(EDGE_FEATURE_DIM, dtype=torch.float)
    for i, dim in enumerate(tensor['dim']):
        h_edge[i] = dim
    
    h_edge_size = torch.zeros(EDGE_FEATURE_DIM, dtype=torch.float)
    h_edge_size[0] = math.prod(tensor['dim'])

    return torch.vstack([h_edge, h_edge_size])

class SubgraphDataset(Dataset):
    def __init__(self, root):   
        super().__init__(root)
        json_files = [f for f in os.listdir(root) if f.endswith('.json') and f.startswith('original_')]

        hash_to_time = json.load(open(os.path.join(root, 'performance.json'), 'r'))

        self.subgraph_files = []
        self.execution_times = []
        for json_file in json_files:
            hash = json_file[len('original_'):-len('.json')]
            if hash in hash_to_time:
                self.subgraph_files.append(json_file)
                self.execution_times.append(hash_to_time[hash])
        
        assert len(self.subgraph_files) == len(self.execution_times)

    def len(self):
        return len(self.subgraph_files)
    
    def get(self, idx):
        json_path = os.path.join(self.root, self.subgraph_files[idx])
        json_graph = json.load(open(json_path, 'r'))

        nodes = []
        edge_index = []
        edge_attr = []

        producer_of = {}

        for node_idx, node in enumerate(json_graph):
            nodes.append(get_node_features(node))

            for t in node['output_tensors']:
                producer_of[t['guid']] = node_idx

        for dst_idx, node in enumerate(json_graph):
            for t in node['input_tensors']:
                src_idx = producer_of[t['guid']]
                edge_index.append([src_idx, dst_idx])
                edge_attr.append(get_edge_features(t))
        
        return Data(
            x=torch.stack(nodes, dim=0),
            edge_index=torch.tensor(edge_index, dtype=torch.long).t().contiguous(),
            edge_attr=torch.stack(edge_attr, dim=0),
            y=torch.tensor([self.execution_times[idx]], dtype=torch.float),
        )

In [ ]:
class SubgraphEncoder(nn.Module):
    """
    Same as your original model up to the graph embedding 'g'.
    Produces a [num_graphs, hidden] representation.
    """
    def __init__(self, node_in=60, edge_in=8, hidden=256, layers=8, dropout=0.2):
        super().__init__()
        self.dropout = dropout
        self.node_in = node_in
        self.edge_in = edge_in

        self.x_proj = nn.Sequential(
            nn.Linear(node_in, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
        )

        def make_mlp():
            return nn.Sequential(
                nn.Linear(hidden, hidden),
                nn.ReLU(inplace=True),
                nn.Linear(hidden, hidden),
            )

        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(GINEConv(make_mlp(), edge_dim=edge_in))
        self.bns.append(nn.BatchNorm1d(hidden))
        for _ in range(layers - 1):
            self.convs.append(GINEConv(make_mlp(), edge_dim=edge_in))
            self.bns.append(nn.BatchNorm1d(hidden))

    @torch.no_grad()
    def embed(self, data):
        """
        Convenience helper that runs in eval/no-grad and returns embeddings.
        """
        was_training = self.training
        self.eval()
        g = self.forward(data, return_embedding=True)
        if was_training:
            self.train()
        return g

    def forward(self, data, return_embedding=False):
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        # Flatten node features [N,6,10] -> [N,60] if needed
        if x.dim() == 3:
            x = x.view(x.size(0), -1)

        # Flatten edge features [E,2,4] -> [E,8] if needed
        if edge_attr is not None and edge_attr.dim() == 3:
            edge_attr = edge_attr.view(edge_attr.size(0), -1)

        x = self.x_proj(x)  # [N, hidden]

        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index, edge_attr)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Graph-level representation
        g = global_mean_pool(x, batch)  # [num_graphs, hidden]
        if return_embedding:
            return g
        return g  # (kept for compatibility)


class SubgraphRegressorXGB(nn.Module):
    """
    Wraps the encoder and uses an XGBoost regressor as the head.
    - Call `fit_xgb(dataloader, y_tensor_or_extractor=...)` to train the XGB head.
    - Forward will return predictions if the XGB head is fitted; otherwise it returns embeddings.
    """
    def __init__(self, node_in=60, edge_in=8, hidden=256, layers=8, dropout=0.2, xgb_kwargs=None):
        super().__init__()
        self.encoder = SubgraphEncoder(node_in, edge_in, hidden, layers, dropout)
        self.xgb = None
        self.xgb_kwargs = xgb_kwargs or dict(
            n_estimators=600,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=42,
        )

    def forward(self, data):
        """
        If XGB is trained, returns predictions [num_graphs].
        If not, returns embeddings [num_graphs, hidden] so you can pretrain/diagnose.
        """
        g = self.encoder.embed(data)  # [B, hidden]
        if self.xgb is None:
            return g  # embeddings
        preds = self._xgb_predict_from_tensor(g)  # [B]
        return preds

    def _xgb_predict_from_tensor(self, g_tensor: torch.Tensor) -> torch.Tensor:
        g_cpu = g_tensor.detach().cpu().numpy()
        yhat = self.xgb.predict(g_cpu)
        return torch.from_numpy(yhat).to(g_tensor.device).float()

    @torch.no_grad()
    def extract_embeddings(self, dataloader, device="cuda" if torch.cuda.is_available() else "cpu"):
        """
        Runs the encoder over a dataloader and returns (X, y) where:
          - X: np.ndarray of shape [num_graphs, hidden]
          - y: np.ndarray of shape [num_graphs]
        Assumes each batch item has `data.y` as the scalar regression target.
        """
        self.to(device)
        was_training = self.training
        self.eval()

        X, Y = [], []
        for batch in dataloader:
            batch = batch.to(device)
            g = self.encoder.embed(batch)  # [b, hidden]
            X.append(g.detach().cpu().numpy())
            if hasattr(batch, "y") and batch.y is not None:
                y = batch.y.view(-1).detach().cpu().numpy()
                Y.append(y)
        X = np.concatenate(X, axis=0)
        Y = np.concatenate(Y, axis=0) if len(Y) else None

        if was_training:
            self.train()
        return X, Y

    def fit_xgb(self, dataloader, device="cuda" if torch.cuda.is_available() else "cpu", X=None, y=None):
        """
        Fit the XGBoost head. You can either:
          - pass a dataloader (we'll extract embeddings + y), or
          - pass precomputed (X, y) numpy arrays.
        """
        if X is None or y is None:
            X, y = self.extract_embeddings(dataloader, device=device)
        self.xgb = xgb.XGBRegressor(**self.xgb_kwargs)
        self.xgb.fit(X, y)
        return self

    def predict_xgb(self, dataloader_or_tensor, device="cuda" if torch.cuda.is_available() else "cpu"):
        """
        Inference helper:
          - If given a dataloader of pyg batches -> returns np.ndarray predictions for the whole set.
          - If given a torch.Tensor of embeddings -> returns np.ndarray predictions directly.
        """
        if self.xgb is None:
            raise RuntimeError("XGBoost head is not fitted. Call fit_xgb(...) first.")
        if isinstance(dataloader_or_tensor, torch.Tensor):
            return self.xgb.predict(dataloader_or_tensor.detach().cpu().numpy())

        # dataloader path
        self.to(device)
        was_training = self.training
        self.eval()

        preds = []
        for batch in dataloader_or_tensor:
            batch = batch.to(device)
            g = self.encoder.embed(batch)
            preds.append(self.xgb.predict(g.detach().cpu().numpy()))
        if was_training:
            self.train()
        return np.concatenate(preds, axis=0)

In [23]:
# --- keep your own imports for pyg dataset/model ---
# from your_module import SubgraphDataset, SubgraphRegressorXGB, SubgraphRegressor

# --- helpers ---
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def _ensure_scalar_preds(model, batch, device):
    """
    Returns scalar predictions as a torch.FloatTensor [B].
    Works for:
      - Pure torch model that directly outputs [B]
      - XGB model where forward() returns embeddings if xgb is None
    """
    out = model(batch)  # could be [B] or [B, hidden]
    if isinstance(out, torch.Tensor):
        # If the model returned embeddings (2D), try to use XGB head if available
        if out.dim() == 2:
            if hasattr(model, "xgb") and model.xgb is not None:
                # run xgb on the embeddings
                g = out  # embeddings
                preds = model._xgb_predict_from_tensor(g)  # [B]
                return preds
            else:
                raise RuntimeError(
                    "Model returned embeddings (2D) but XGBoost head is not fitted yet. "
                    "Call model.fit_xgb(...) before evaluation."
                )
        # 1D tensor: good
        return out.view(-1).float()
    # Fallback (shouldn't happen): convert to tensor
    return torch.as_tensor(out, dtype=torch.float32, device=device).view(-1)


def evaluate(model, loader, device):
    model.eval()
    all_pred, all_true = [], []
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = _ensure_scalar_preds(model, batch, device)  # [B]
            all_pred.append(pred)
            all_true.append(batch.y.view(-1).to(device))

    if not all_pred:
        return float("nan"), float("nan"), float("nan")
    preds = torch.cat(all_pred)
    trues = torch.cat(all_true)
    mae = (preds - trues).abs().mean().item()
    rmse = torch.sqrt(((preds - trues) ** 2).mean()).item()
    return mae, mae, rmse  # (val_loss proxy, MAE, RMSE)


@torch.no_grad()
def pairwise_accuracy(preds: torch.Tensor, trues: torch.Tensor, exclude_ties: bool = True):
    n = preds.numel()
    if n < 2:
        return float('nan'), 0, 0
    pdiff = preds.view(n, 1) - preds.view(1, n)
    ydiff = trues.view(n, 1) - trues.view(1, n)
    tri_mask = torch.triu(torch.ones(n, n, dtype=torch.bool, device=preds.device), diagonal=1)
    comp_mask = tri_mask & (ydiff != 0) if exclude_ties else tri_mask
    total = int(comp_mask.sum().item())
    if total == 0:
        return float('nan'), 0, 0
    correct = ((torch.sign(pdiff) == torch.sign(ydiff)) & comp_mask).sum().item()
    return correct / total, correct, total


def _save_checkpoint_xgb(model, base_path):
    """
    Saves:
      - encoder weights to f"{base_path}.pt"
      - xgboost model to f"{base_path}.json"
    """
    enc_path = f"{base_path}.pt"
    xgb_path = f"{base_path}.json"
    torch.save(model.encoder.state_dict(), enc_path)
    if hasattr(model, "xgb") and model.xgb is not None:
        model.xgb.save_model(xgb_path)
    return enc_path, xgb_path


def _load_checkpoint_xgb(model, base_path, device):
    enc_path = f"{base_path}.pt"
    xgb_path = f"{base_path}.json"
    model.encoder.load_state_dict(torch.load(enc_path, map_location=device))
    if os.path.exists(xgb_path):
        model.xgb = None  # ensure a fresh XGB object
        # XGBRegressor().load_model maintains hyperparams inside file
        import xgboost as xgb
        model.xgb = xgb.XGBRegressor()
        model.xgb.load_model(xgb_path)


def train_one(model, train_loader, val_loader, device, ckpt_path):
    """
    Unified trainer:
      - If the model has an XGBoost head (SubgraphRegressorXGB), we:
          1) (Optionally) train encoder with zero or user-provided loop (skipped here for simplicity)
          2) Fit the XGB head on frozen encoder embeddings
          3) Evaluate on val
          4) Save encoder + XGB head

      - Otherwise, we run the original gradient training loop.
    """
    # Path stem (we’ll add .pt/.json inside)
    base_ckpt = os.path.splitext(ckpt_path)[0]

    model.to(device)
    # Fit xgboost on current encoder embeddings
    model.fit_xgb(train_loader, device=device)

    # Evaluate once after fit
    val_loss, val_mae, val_rmse = evaluate(model, val_loader, device)
    _save_checkpoint_xgb(model, base_ckpt)
    print(f"[XGB] Val MAE: {val_mae:.4f} | Val RMSE: {val_rmse:.4f}")
    # Use MAE as our "best" (to be consistent with your return contract)
    return val_loss

def train_eval_grid(
    data_root: str = "./data",
    batch_size: int = 16,
    seed: int = 42,
    hidden_grid = (64, 128, 256),
    layers_grid = (2, 3, 4),
    model_prefix: str = "test",
    n_splits: int = 5,          # K-fold splits
    shuffle_folds: bool = True, # whether to shuffle before splitting
):
    """
    Grid search with sklearn KFold CV for the XGBoost-head model.
    Selection = mean validation MAE across folds.
    After selecting best config, compute OOF predictions across all folds and
    report OOF MAE/RMSE.
    """
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---- load dataset ----
    dataset = SubgraphDataset(data_root)
    n_total = len(dataset)
    assert n_total >= n_splits, f"Need at least {n_splits} samples for {n_splits}-fold CV."

    kf = KFold(n_splits=n_splits, shuffle=shuffle_folds, random_state=seed)
    folds = [(train_idx.tolist(), val_idx.tolist()) for train_idx, val_idx in kf.split(range(n_total))]

    best_cfg = None
    best_cv_val_mae = float('inf')
    best_cfg_ckpt = None  # list of fold checkpoint stems (without extension)

    print(f"\nUsing sklearn KFold: {n_splits} folds over {n_total} samples.")

    for hidden, layers in product(hidden_grid, layers_grid):
        print(f"\n=== Grid config: hidden={hidden}, layers={layers} ===")

        fold_val_maes, fold_val_rmses, fold_val_losses = [], [], []
        fold_best_ckpts = []

        for fold_id, (train_idx, val_idx) in enumerate(folds, start=1):
            train_subset = torch.utils.data.Subset(dataset, train_idx)
            val_subset   = torch.utils.data.Subset(dataset, val_idx)

            train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
            val_loader   = DataLoader(val_subset,   batch_size=batch_size, shuffle=False)

            # Use the XGB head model
            model = SubgraphRegressorXGB(
                node_in=60, edge_in=8, hidden=hidden, layers=layers, dropout=0.2
            ).to(device)

            ckpt_stem = f"_cv_tmp_h{hidden}_L{layers}_fold{fold_id}"
            _ = train_one(model, train_loader, val_loader, device, ckpt_path=ckpt_stem)

            # Reload encoder + XGB head and evaluate
            re_model = SubgraphRegressorXGB(
                node_in=60, edge_in=8, hidden=hidden, layers=layers, dropout=0.2
            ).to(device)
            _load_checkpoint_xgb(re_model, ckpt_stem, device=device)
            v_loss, v_mae, v_rmse = evaluate(re_model, val_loader, device)
            print(f"  Fold {fold_id}/{n_splits} → Val MAE: {v_mae:.4f} | Val RMSE: {v_rmse:.4f} | Val Loss: {v_loss:.4f}")

            fold_val_losses.append(v_loss)
            fold_val_maes.append(v_mae)
            fold_val_rmses.append(v_rmse)
            fold_best_ckpts.append(ckpt_stem)

        mean_mae = float(np.mean(fold_val_maes))
        mean_rmse = float(np.mean(fold_val_rmses))
        mean_loss = float(np.mean(fold_val_losses))
        print(f"Config result (mean over {n_splits} folds) → Val MAE: {mean_mae:.4f} | Val RMSE: {mean_rmse:.4f} | Val Loss: {mean_loss:.4f}")

        if mean_mae < best_cv_val_mae:
            best_cv_val_mae = mean_mae
            best_cfg = {"hidden": hidden, "layers": layers}
            best_cfg_ckpt = fold_best_ckpts
            print("  ✔ New best config so far.")

    assert best_cfg is not None, "Grid search did not evaluate any configuration."
    print(f"\nBest config found: hidden={best_cfg['hidden']}, layers={best_cfg['layers']} (CV Val MAE={best_cv_val_mae:.4f})")

    # ---- OOF predictions with the best config ----
    all_oof_preds, all_oof_trues = [], []
    for fold_id, (_, val_idx) in enumerate(folds, start=1):
        val_subset = torch.utils.data.Subset(dataset, val_idx)
        val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

        ckpt_stem = best_cfg_ckpt[fold_id - 1]
        model = SubgraphRegressorXGB(node_in=60, edge_in=8,
                                     hidden=best_cfg["hidden"],
                                     layers=best_cfg["layers"],
                                     dropout=0.2).to(device)
        _load_checkpoint_xgb(model, ckpt_stem, device=device)
        model.eval()

        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                # ensure scalar preds
                preds = _ensure_scalar_preds(model, batch, device)
                all_oof_preds.append(preds)
                all_oof_trues.append(batch.y.view(-1).to(device))

    oof_preds = torch.cat(all_oof_preds)
    oof_trues = torch.cat(all_oof_trues)
    oof_mae = (oof_preds - oof_trues).abs().mean().item()
    oof_rmse = torch.sqrt(((oof_preds - oof_trues) ** 2).mean()).item()

    print(f"\n🏁 CV results (out-of-fold): MAE={oof_mae:.6f} | RMSE={oof_rmse:.6f}")

    pair_acc, correct, total = pairwise_accuracy(oof_preds, oof_trues, exclude_ties=True)
    print(f"🔁 Pairwise comparison accuracy (OOF): {pair_acc:.4f} ({correct}/{total} comparable pairs)")

    # ---- Optional final artifact: fit on (almost) all data ----
    all_indices = set(range(n_total))
    final_val_idx = set(folds[-1][1])  # keep a tiny val for sanity checks
    final_train_idx = [i for i in all_indices if i not in final_val_idx]

    final_train_subset = torch.utils.data.Subset(dataset, final_train_idx)
    final_val_subset   = torch.utils.data.Subset(dataset, list(final_val_idx))

    final_train_loader = DataLoader(final_train_subset, batch_size=batch_size, shuffle=True)
    final_val_loader   = DataLoader(final_val_subset,   batch_size=batch_size, shuffle=False)

    final_model = SubgraphRegressorXGB(node_in=60, edge_in=8,
                                       hidden=best_cfg["hidden"],
                                       layers=best_cfg["layers"],
                                       dropout=0.2).to(device)

    model_out_stem = f"{model_prefix}_hidden{best_cfg['hidden']}_layers{best_cfg['layers']}"
    _ = train_one(final_model, final_train_loader, final_val_loader, device, ckpt_path=model_out_stem)
    print(f"  ✔ Final model for best config saved to {model_out_stem}.pt/.json")

    return {
        "best_config": best_cfg,
        "val_mae": best_cv_val_mae,  # mean CV val MAE used for selection
        "test_mae": oof_mae,         # OOF MAE (proxy for held-out test)
        "test_rmse": oof_rmse,       # OOF RMSE (proxy for held-out test)
        "checkpoint_stem": model_out_stem,  # encoder (.pt) + xgb (.json)
    }

In [24]:
train_eval_grid(data_root='/home/kitao/projects/mirage/dataset/10_16_25_cleaned',
           model_prefix="models/10_29_exec_time_gine",
           hidden_grid=[128, 256, 512],
           layers_grid=[8, 16, 32])


Using sklearn KFold: 5 folds over 233 samples.

=== Grid config: hidden=128, layers=8 ===
[XGB] Val MAE: 0.0099 | Val RMSE: 0.0199
  Fold 1/5 → Val MAE: 0.0099 | Val RMSE: 0.0199 | Val Loss: 0.0099
[XGB] Val MAE: 0.0095 | Val RMSE: 0.0200
  Fold 2/5 → Val MAE: 0.0095 | Val RMSE: 0.0200 | Val Loss: 0.0095
[XGB] Val MAE: 0.0120 | Val RMSE: 0.0253
  Fold 3/5 → Val MAE: 0.0120 | Val RMSE: 0.0253 | Val Loss: 0.0120
[XGB] Val MAE: 0.0091 | Val RMSE: 0.0148
  Fold 4/5 → Val MAE: 0.0091 | Val RMSE: 0.0148 | Val Loss: 0.0091
[XGB] Val MAE: 0.0096 | Val RMSE: 0.0242
  Fold 5/5 → Val MAE: 0.0096 | Val RMSE: 0.0242 | Val Loss: 0.0096
Config result (mean over 5 folds) → Val MAE: 0.0100 | Val RMSE: 0.0208 | Val Loss: 0.0100
  ✔ New best config so far.

=== Grid config: hidden=128, layers=16 ===
[XGB] Val MAE: 0.0095 | Val RMSE: 0.0188
  Fold 1/5 → Val MAE: 0.0095 | Val RMSE: 0.0188 | Val Loss: 0.0095
[XGB] Val MAE: 0.0139 | Val RMSE: 0.0254
  Fold 2/5 → Val MAE: 0.0139 | Val RMSE: 0.0254 | Val Loss

{'best_config': {'hidden': 256, 'layers': 8},
 'val_mae': 0.009236199222505093,
 'test_mae': 0.00923454575240612,
 'test_rmse': 0.01748318038880825,
 'checkpoint_stem': 'models/10_29_exec_time_gine_hidden256_layers8'}